# NanoReason training on Kaggle T4

A resumable three-session run: **baseline + SFT → GRPO → final evaluation + export**.

Before each run, enable **GPU T4 ×2** and **Internet** in notebook settings. NanoReason uses one T4 because Phi-3 fits on one 16 GB GPU and this TRL pipeline does not safely split across both cards.

## 1. Choose this session's stage

Run one stage per saved notebook version. For sessions 2 and 3, attach the preceding version's output using **Add Input → Your Work**.

In [ ]:
RUN_STAGE = 'sft'  # choose: 'sft', 'grpo', 'evaluate', or 'all'
assert RUN_STAGE in {'sft', 'grpo', 'evaluate', 'all'}

## 2. Get the code

In [ ]:
import glob, os, shutil
DST = '/kaggle/working/nanoreason'
if os.path.exists(DST):
    shutil.rmtree(DST)
candidates = glob.glob('/kaggle/input/**/pyproject.toml', recursive=True)
candidates.sort(key=lambda value: value.count('/'))
if candidates:
    source = os.path.dirname(candidates[0]); shutil.copytree(source, DST); print('Copied repository from', source)
else:
    if os.system('git clone https://github.com/pixeLatedbLue/NanoReason.git ' + DST) != 0:
        raise RuntimeError('Could not clone NanoReason; check that Internet is enabled')
os.chdir(DST)
print('cwd =', os.getcwd())

## 3. Restore an earlier session, when attached

In [ ]:
from pathlib import Path
def merge_tree(source: Path, destination: Path) -> None:
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True); print('Restored', source, 'to', destination)
input_root = Path('/kaggle/input'); artifact_roots = set()
for marker in list(input_root.glob('**/artifacts/**/adapter_config.json')) + list(input_root.glob('**/artifacts/grpo/checkpoint-*/trainer_state.json')):
    parts = marker.parts; artifact_roots.add(Path(*parts[:parts.index('artifacts') + 1]))
for artifact_root in sorted(artifact_roots):
    merge_tree(artifact_root, Path('/kaggle/working/artifacts'))
    merge_tree(artifact_root.parent / 'results', Path('/kaggle/working/results'))
if not artifact_roots: print('No earlier NanoReason output attached; starting without a restore')

## 4. Install and verify the environment

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!pip -q install "transformers>=4.46,<4.47" "trl==0.14.0" "peft>=0.12,<0.14" \
                "datasets>=2.21,<3.0" "accelerate>=0.34,<1.0" "bitsandbytes>=0.43,<0.45"
!pip -q install -e . --no-deps
!pip -q uninstall -y torchvision torchaudio
import torch
print('torch:', torch.__version__); print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Kaggle GPU is not enabled'
!python -m unittest discover -s tests

## 5. Session 1 — baseline evaluation and SFT

In [ ]:
from pathlib import Path
import subprocess
def run(command: list[str]) -> None:
    print('>', ' '.join(command)); subprocess.run(command, check=True)
if RUN_STAGE in {'sft', 'all'}:
    if not Path('/kaggle/working/results/baseline.json').exists():
        run(['python','-m','nanoreason.evaluate','--config','configs/kaggle_t4.toml','--run-name','baseline'])
    if not Path('/kaggle/working/artifacts/sft-final/adapter_config.json').exists():
        checkpoints=sorted(Path('/kaggle/working/artifacts/sft').glob('checkpoint-*'), key=lambda x:int(x.name.rsplit('-',1)[-1]))
        command=['python','-m','nanoreason.train_sft','--config','configs/kaggle_t4.toml']
        if checkpoints: command += ['--resume-from-checkpoint',str(checkpoints[-1])]
        run(command)
    else: print('SFT adapter already exists; skipping training')

## 6. Session 2 — GRPO

In [ ]:
if RUN_STAGE in {'grpo', 'all'}:
    if not Path('/kaggle/working/artifacts/sft-final/adapter_config.json').exists():
        raise FileNotFoundError('Attach the Session 1 output before running GRPO')
    if not Path('/kaggle/working/artifacts/grpo-final/adapter_config.json').exists():
        checkpoints=sorted(Path('/kaggle/working/artifacts/grpo').glob('checkpoint-*'), key=lambda x:int(x.name.rsplit('-',1)[-1]))
        command=['python','-m','nanoreason.train_grpo','--config','configs/kaggle_t4.toml']
        if checkpoints: command += ['--resume-from-checkpoint',str(checkpoints[-1])]
        run(command)
    else: print('GRPO adapter already exists; skipping training')

## 7. Session 3 — evaluation, comparison, and export

In [ ]:
if RUN_STAGE in {'evaluate', 'all'}:
    adapter='/kaggle/working/artifacts/grpo-final'
    if not Path(adapter,'adapter_config.json').exists(): raise FileNotFoundError('Attach the Session 2 output before evaluating')
    run(['python','-m','nanoreason.evaluate','--config','configs/kaggle_t4.toml','--adapter',adapter,'--run-name','grpo_final'])
    run(['python','-m','nanoreason.compare_results','--baseline','/kaggle/working/results/baseline.json','--candidate','/kaggle/working/results/grpo_final.json','--ci','--paired'])
    run(['python','-m','nanoreason.export','--config','configs/kaggle_t4.toml','--adapter',adapter,'--merged-dir','/kaggle/working/artifacts/merged','--load-4bit','--prompt','A pen costs 3 dollars. How much for 4 pens?'])

## 8. Save and continue

After a stage completes, save a notebook version and attach that version's output to the next session. The notebook restores the latest checkpoints automatically.

Expected outputs: `results/baseline.json`, `artifacts/sft-final/`, `artifacts/grpo-final/`, `results/grpo_final.json`, `results/forgetting_probe.json`, and `artifacts/merged/`.